# AROHA · Aircraft engineering workbench

**Illustrative inputs only.** This notebook demonstrates the calculation workflow, not that the actual aircraft will fly. All inputs use SI units unless explicitly labelled. Edit `configs/demo.json` using measured data and track every assumption.

Use Shift+Enter to run each cell. The starter layout is two front tilting rotors plus one fixed vertical rear rotor; it is not a confirmed aircraft configuration.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aroha.physics import *
from aroha.integrations import adrpy_cruise, suave_vehicle
root = Path.cwd()
if not (root / 'configs').exists(): root = root.parent
c = load_config(root / 'configs/demo.json')
pd.DataFrame({'assumption': c['assumptions']})

## Hover load allocation

Solve vertical force, roll and pitch moment balance about the CG. Report yaw separately. Three scalar thrust controls cannot independently span four hover control channels.

In [ ]:
h = hover_allocation(c)
pd.DataFrame({'rotor':[r['name'] for r in c['rotors']], 'required_thrust_n':h['thrust_n'], 'assumed_limit_n':[r['max_thrust_n'] for r in c['rotors']]})

In [ ]:
{'yaw_residual_nm':h['yaw_residual_nm'], 'four_channel_rank':h['four_channel_rank'], 'hover_control_validated':False}

## Fixed-wing sizing and ADRpy

Minimum wing area from the chosen stall limit uses an assumed CLmax. ADRpy uses its own induced-drag estimate; the equation reconstruction below checks integration and units, not experimental accuracy.

In [ ]:
rho = density(c)
w = c['mass_kg'] * G
minimum_area = 2*w/(rho*c['stall_limit_mps']**2*c['cl_max'])
ws = np.linspace(20,150,100)
adr, reconstructed = adrpy_cruise(c,ws)
np.testing.assert_allclose(adr,reconstructed,rtol=1e-12)
print(f'Minimum area at assumed stall requirement: {minimum_area:.3f} m²')
plt.plot(ws,adr)
plt.xlabel('Wing loading W/S [N/m²]'); plt.ylabel('ADRpy cruise T/W [–]'); plt.grid(alpha=.2); plt.show()

## SUAVE backend

This creates a vehicle/wing and three rotor geometry objects, and queries the atmosphere. The motor/ESC/battery networks, blade geometry, control surfaces, mission solver and transition control law still require aircraft-specific inputs.

In [ ]:
vehicle, rotors, suave_rho = suave_vehicle(c)
assert len(rotors)==3
print(vehicle.tag, [r.tag for r in rotors])
print('ADRpy vs SUAVE density:', rho, suave_rho)

## Mass sensitivity

Same geometry, atmosphere and assumed efficiencies. This is a parameter sweep, not a confidence interval.

In [ ]:
import copy
rows=[]
for mass in np.linspace(.8*c['mass_kg'],1.2*c['mass_kg'],9):
    trial=copy.deepcopy(c); trial['mass_kg']=float(mass)
    hh=hover_allocation(trial)
    p=hover_power(hh['thrust_n'],[r['diameter_m'] for r in trial['rotors']],rho,c['hover_figure_of_merit'],c['motor_esc_efficiency']).sum()+c['avionics_power_w']
    rows.append({'mass_kg':mass,'hover_power_w':p,'stall_speed_mps':stall_speed(mass*G,rho,c['wing_area_m2'],c['cl_max'])})
pd.DataFrame(rows).set_index('mass_kg').plot(subplots=True,figsize=(8,5),grid=True); plt.show()

## Reduced transition screen

Tilt is measured from vertical. This solves only Fx, Fz, roll and pitch with bounded rotor thrust and wing lift. No side-force/yaw balance, rotor reaction torque, aerodynamic pitching moment, interference, transient dynamics or actuator-rate checks are included.

In [ ]:
for speed,tilt in [(0,0),(0,90),(20,0),(20,90)]:
    result=transition_screen(c,speed,tilt)
    print(speed,tilt,result['reduced_balance_found'],result['flight_feasibility'])

## Next analyses

Run `python scripts/run.py analyse` for the reproducible plot/CSV/JSON report. Run `python scripts/run.py vsp` for the independent reference-wing solver and mesh check. This notebook intentionally does not imply that a reference-wing result represents the manta-inspired UAV. Replace assumptions with CAD, mass/CG measurements, propeller test maps, airfoil data, actuator and battery limits before constructing the actual aircraft model.